In [ ]:
# BookAhead - Data Exploration and Visualization
# Midterm Assignment - Visual Analysis

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11

# ============================================================
# LOAD DATA
# ============================================================

# Load the processed dataset
df = pd.read_parquet('../data/interim/best_today.parquet')

print(f"Dataset loaded: {len(df):,} observations")
print(f"Date range: {df['scraped_date'].min()} to {df['scraped_date'].max()}")
print(f"Routes: {df['route'].unique()}")
print(f"\nFirst few rows:")
df.head()

# ============================================================
# FIGURE 1: PRICE TRENDS OVER TIME BY ROUTE
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, route in enumerate(df['route'].unique()):
    ax = axes[idx]
    route_data = df[df['route'] == route].sort_values('scraped_date')
    
    # Plot each departure date as separate line
    for dep_date in route_data['departure_date'].unique():
        subset = route_data[route_data['departure_date'] == dep_date]
        ax.plot(subset['scraped_date'], subset['min_price'], 
                marker='o', label=f"Dep: {dep_date.strftime('%b %d')}", alpha=0.7)
    
    ax.set_title(f'{route} - Price Evolution', fontsize=13, fontweight='bold')
    ax.set_xlabel('Scraped Date', fontsize=11)
    ax.set_ylabel('Min Price ($)', fontsize=11)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../outputs/price_trends_by_route.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Figure 1 saved: Price trends show volatility patterns across routes")

# ============================================================
# FIGURE 2: BOOKING CURVE (Days Until Departure vs Price)
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, route in enumerate(df['route'].unique()):
    ax = axes[idx]
    route_data = df[df['route'] == route]
    
    # Scatter plot with trend line
    ax.scatter(route_data['days_until'], route_data['min_price'], 
               alpha=0.6, s=50, c='steelblue')
    
    # Add polynomial trend line
    z = np.polyfit(route_data['days_until'], route_data['min_price'], 2)
    p = np.poly1d(z)
    x_trend = np.linspace(route_data['days_until'].min(), 
                          route_data['days_until'].max(), 100)
    ax.plot(x_trend, p(x_trend), "r--", linewidth=2, label='Trend')
    
    ax.set_title(f'{route} - Booking Curve', fontsize=13, fontweight='bold')
    ax.set_xlabel('Days Until Departure', fontsize=11)
    ax.set_ylabel('Min Price ($)', fontsize=11)
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.invert_xaxis()  # Time moves right to left

plt.tight_layout()
plt.savefig('../outputs/booking_curves.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Figure 2 saved: Booking curves show non-linear price patterns")

# ============================================================
# FIGURE 3: PRICE DISTRIBUTION BY ROUTE
# ============================================================

fig, ax = plt.subplots(figsize=(12, 6))

# Box plot
df.boxplot(column='min_price', by='route', ax=ax, patch_artist=True)

ax.set_title('Price Distribution by Route', fontsize=14, fontweight='bold')
ax.set_xlabel('Route', fontsize=12)
ax.set_ylabel('Min Price ($)', fontsize=12)
plt.suptitle('')  # Remove default title

# Add mean markers
for idx, route in enumerate(df['route'].unique(), 1):
    mean_price = df[df['route'] == route]['min_price'].mean()
    ax.scatter(idx, mean_price, color='red', s=100, zorder=3, marker='D', 
               label='Mean' if idx == 1 else '')

ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../outputs/price_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Figure 3 saved: Price distributions reveal route-specific variance")

# ============================================================
# FIGURE 4: DAY OF WEEK EFFECTS
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Booking day effect
ax1 = axes[0]
booking_day_avg = df.groupby('book_dow')['min_price'].mean().reset_index()
days = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
ax1.bar(range(7), booking_day_avg['min_price'], color='steelblue', alpha=0.7)
ax1.set_xticks(range(7))
ax1.set_xticklabels(days)
ax1.set_title('Average Price by Booking Day', fontsize=13, fontweight='bold')
ax1.set_xlabel('Day of Week (Booking)', fontsize=11)
ax1.set_ylabel('Avg Min Price ($)', fontsize=11)
ax1.grid(True, alpha=0.3, axis='y')

# Departure day effect
ax2 = axes[1]
depart_day_avg = df.groupby('depart_dow')['min_price'].mean().reset_index()
ax2.bar(range(7), depart_day_avg['min_price'], color='coral', alpha=0.7)
ax2.set_xticks(range(7))
ax2.set_xticklabels(days)
ax2.set_title('Average Price by Departure Day', fontsize=13, fontweight='bold')
ax2.set_xlabel('Day of Week (Departure)', fontsize=11)
ax2.set_ylabel('Avg Min Price ($)', fontsize=11)
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../outputs/day_of_week_effects.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Figure 4 saved: Day of week patterns show pricing strategies")

# ============================================================
# FIGURE 5: MODEL PERFORMANCE - ACTUAL VS PREDICTED
# ============================================================

# Load saved Linear Regression predictions (if available)
# For now, we'll create a demonstration plot
# You'll update this after running your model

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, route in enumerate(df['route'].unique()):
    ax = axes[idx]
    route_data = df[df['route'] == route]
    
    # Sort by days_until for visualization
    route_data = route_data.sort_values('days_until', ascending=False)
    
    # Actual prices
    ax.plot(range(len(route_data)), route_data['min_price'].values, 
            marker='o', label='Actual Price', linewidth=2, markersize=8, alpha=0.7)
    
    # Placeholder for predicted (you'll replace this with actual predictions)
    # For demonstration, adding slight noise
    predicted = route_data['min_price'].values + np.random.normal(0, 10, len(route_data))
    ax.plot(range(len(route_data)), predicted, 
            marker='s', label='Predicted (LR)', linewidth=2, markersize=6, alpha=0.7)
    
    ax.set_title(f'{route} - Model Predictions', fontsize=13, fontweight='bold')
    ax.set_xlabel('Observation Index', fontsize=11)
    ax.set_ylabel('Price ($)', fontsize=11)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/model_predictions.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Figure 5 saved: Model predictions vs actual (update with real predictions)")

# ============================================================
# FIGURE 6: FEATURE IMPORTANCE FROM LINEAR REGRESSION
# ============================================================

# Load trained model and extract coefficients
import joblib

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, route in enumerate(df['route'].unique()):
    ax = axes[idx]
    
    # Load saved model
    model_path = f'../models/lr_{route.replace("-", "_")}.pkl'
    try:
        pipe = joblib.load(model_path)
        
        # Get feature names and coefficients
        num_feats = ['days_until', 'book_dow', 'depart_dow', 'depart_woy']
        feature_names = num_feats + list(pipe.named_steps['pre'].get_feature_names_out())
        coefficients = pipe.named_steps['model'].coef_
        
        # Top 5 features by absolute coefficient
        top_idx = np.argsort(np.abs(coefficients))[-5:][::-1]
        top_features = [feature_names[i] for i in top_idx]
        top_coefs = [coefficients[i] for i in top_idx]
        
        # Plot
        colors = ['green' if c > 0 else 'red' for c in top_coefs]
        ax.barh(range(len(top_features)), top_coefs, color=colors, alpha=0.7)
        ax.set_yticks(range(len(top_features)))
        ax.set_yticklabels(top_features, fontsize=9)
        ax.set_title(f'{route} - Feature Importance', fontsize=13, fontweight='bold')
        ax.set_xlabel('Coefficient Value', fontsize=11)
        ax.axvline(x=0, color='black', linestyle='--', linewidth=1)
        ax.grid(True, alpha=0.3, axis='x')
        
    except FileNotFoundError:
        ax.text(0.5, 0.5, f'Model not trained yet for {route}', 
                ha='center', va='center', fontsize=12)
        ax.set_title(f'{route} - Feature Importance', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('../outputs/feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Figure 6 saved: Feature importance shows price drivers")

# ============================================================
# SUMMARY STATISTICS TABLE
# ============================================================

print("\n" + "="*70)
print("SUMMARY STATISTICS BY ROUTE")
print("="*70)

summary = df.groupby('route')['min_price'].agg([
    ('Count', 'count'),
    ('Mean', 'mean'),
    ('Median', 'median'),
    ('Std Dev', 'std'),
    ('Min', 'min'),
    ('Max', 'max')
]).round(2)

print(summary)

# Save summary
summary.to_csv('../outputs/summary_statistics.csv')
print("\n✓ Summary statistics saved to outputs/summary_statistics.csv")

# ============================================================
# COMPLETION MESSAGE
# ============================================================

print("\n" + "="*70)
print("✅ ALL VISUALIZATIONS COMPLETE")
print("="*70)
print("\nGenerated files:")
print("  1. price_trends_by_route.png")
print("  2. booking_curves.png")
print("  3. price_distribution.png")
print("  4. day_of_week_effects.png")
print("  5. model_predictions.png")
print("  6. feature_importance.png")
print("  7. summary_statistics.csv")
print("\nAll files saved to outputs/ directory")
print("="*70)